In [7]:
%pip install xlsxwriter

In [17]:
import pandas as pd
import xlsxwriter
from pathlib import Path

def main():
    # =========================================================================
    # 1) LECTURA DE DATOS CRUDOS DESDE CSV
    # =========================================================================
    csv_path = r"C:\Users\mkoro\Desktop\Transformers\test_dataset\outputs\results\results_completo.csv"  # Ajusta la ruta
    df = pd.read_csv(csv_path)

    # =========================================================================
    # 2) CONFIGURAR ARCHIVO EXCEL DE SALIDA
    # =========================================================================
    excel_filename = "Analisis_XIV.xlsx"
    workbook = xlsxwriter.Workbook(excel_filename)

    # =========================================================================
    # 3) HOJA "Raw Data": DATOS CRUDOS + FÓRMULAS EN ESPAÑOL
    # =========================================================================
    raw_sheet = workbook.add_worksheet("Raw Data")
    
    # Encabezados: columnas originales + "Total Tests", "Accuracy", "Numeric Complexity"
    headers = list(df.columns) + ["Total Tests", "Accuracy", "Numeric Complexity"]
    for col_idx, header in enumerate(headers):
        raw_sheet.write(0, col_idx, header)

    # Se asume el siguiente orden (ajusta si tu CSV difiere):
    # A: ID, B: model, C: temperature, D: top_p, E: top_k
    # F: code, G: result, H: true_count, I: false_count
    # J: execution_time, K: memory_usage_MB, L: algorithmic_complexity
    # Agregaremos:
    # M: Total Tests => =H + I
    # N: Accuracy => =SI(M>0, H/M, "")
    # O: Numeric Complexity => extracción de "loops: X, conditionals: Y" y suma de X + Y

    for r in range(len(df)):
        # Copiar las columnas originales
        for c in range(len(df.columns)):
            raw_sheet.write(r+1, c, df.iat[r, c])
        
        fila_excel = r + 2  # Fila en Excel (encabezado en 1)
        celda_true  = f"H{fila_excel}"  # true_count
        celda_false = f"I{fila_excel}"  # false_count
        celda_total = f"M{fila_excel}"  # Total Tests
        celda_acc   = f"N{fila_excel}"  # Accuracy
        celda_algo  = f"L{fila_excel}"  # algorithmic_complexity (texto)

        # Fórmula para Total Tests (columna M)
        formula_total_tests = f"={celda_true}+{celda_false}"
        raw_sheet.write_formula(r+1, 12, formula_total_tests)

        # Fórmula para Accuracy (columna N)
        # =SI(M2>0, H2/M2, "")
        formula_accuracy = f"=SI({celda_total}>0,{celda_true}/{celda_total},\"\")"
        raw_sheet.write_formula(r+1, 13, formula_accuracy)

        # Fórmula para Numeric Complexity (columna O)
        # Asumiendo que L2 contiene algo como "loops: 3, conditionals: 2"
        # Se usan funciones en español: VALOR, ESPACIOS, EXTRAE, HALLAR, LARGO
        # Ejemplo:
        # =VALOR(ESPACIOS(EXTRAE(L2;HALLAR(":";L2)+1;HALLAR(",";L2)-HALLAR(":";L2)-1)))
        # +VALOR(ESPACIOS(EXTRAE(L2;HALLAR("conditionals:";L2)+13;LARGO(L2))))
        formula_complex = (
            f"=VALOR(ESPACIOS(EXTRAE({celda_algo};HALLAR(\":\";{celda_algo})+1;"
            f"HALLAR(\",\";{celda_algo})-HALLAR(\":\";{celda_algo})-1)))"
            f"+VALOR(ESPACIOS(EXTRAE({celda_algo};HALLAR(\"conditionals:\";{celda_algo})+13;"
            f"LARGO({celda_algo})))))"
        )
        raw_sheet.write_formula(r+1, 14, formula_complex)

    # Ajustar anchos de columna (opcional)
    raw_sheet.set_column(0, 0, 6)    # ID
    raw_sheet.set_column(1, 1, 18)   # model
    raw_sheet.set_column(2, 4, 10)   # temperature, top_p, top_k
    raw_sheet.set_column(5, 5, 60)   # code
    raw_sheet.set_column(6, 6, 25)   # result
    raw_sheet.set_column(7, 8, 12)   # true_count, false_count
    raw_sheet.set_column(9, 10, 14)  # execution_time, memory_usage_MB
    raw_sheet.set_column(11, 11, 25) # algorithmic_complexity
    raw_sheet.set_column(12, 13, 15) # Total Tests, Accuracy
    raw_sheet.set_column(14, 14, 20) # Numeric Complexity

    # =========================================================================
    # 4) HOJA "Comparisons": COMPARACIÓN DE CONFIGURACIONES
    #    Usando funciones en español (PROMEDIO.SI.CONJUNTO, MIN.SI.CONJUNTO, JERARQUIA.EQV)
    # =========================================================================
    comp_sheet = workbook.add_worksheet("Comparisons")
    comp_headers = [
        "Model", "Temperature", "top_p", "top_k",
        "Avg Accuracy", "Avg Time (s)", "Avg Memory (MB)", "Avg Complexity",
        "Config Label",
        "Rank Accuracy", "Rank Time", "Rank Memory", "Rank Complexity",
        "Composite Rank", "Global Best?", "Best per Model?"
    ]
    for col_idx, header in enumerate(comp_headers):
        comp_sheet.write(0, col_idx, header)

    # Obtenemos las configuraciones únicas
    unique_configs = df[['model','temperature','top_p','top_k']].drop_duplicates().sort_values(by=['model'])
    total_configs = len(unique_configs)

    for i, row in enumerate(unique_configs.itertuples(), start=1):
        # Fila en la hoja Comparisons
        fila_excel = i + 0  # La fila 1 es encabezado, i+0 => la primera config en la fila 1
        model_val = row.model
        temp_val  = row.temperature
        top_p_val = row.top_p
        top_k_val = row.top_k

        # Escribir las primeras 4 columnas (A-D)
        comp_sheet.write(fila_excel, 0, model_val)
        comp_sheet.write(fila_excel, 1, temp_val)
        comp_sheet.write(fila_excel, 2, top_p_val)
        comp_sheet.write(fila_excel, 3, top_k_val)

        # Columna E (Avg Accuracy) => PROMEDIO.SI.CONJUNTO en español
        # =PROMEDIO.SI.CONJUNTO('Raw Data'!$N:$N, 'Raw Data'!$B:$B, "=model_val", ...)
        formula_avg_acc = (
            f"=PROMEDIO.SI.CONJUNTO('Raw Data'!$N:$N,"
            f"'Raw Data'!$B:$B,\"{model_val}\","
            f"'Raw Data'!$C:$C,\"{temp_val}\","
            f"'Raw Data'!$D:$D,\"{top_p_val}\","
            f"'Raw Data'!$E:$E,\"{top_k_val}\")"
        )
        comp_sheet.write_formula(fila_excel, 4, formula_avg_acc)

        # Columna F (Avg Time) => 'Raw Data'!$J:$J
        formula_avg_time = (
            f"=PROMEDIO.SI.CONJUNTO('Raw Data'!$J:$J,"
            f"'Raw Data'!$B:$B,\"{model_val}\","
            f"'Raw Data'!$C:$C,\"{temp_val}\","
            f"'Raw Data'!$D:$D,\"{top_p_val}\","
            f"'Raw Data'!$E:$E,\"{top_k_val}\")"
        )
        comp_sheet.write_formula(fila_excel, 5, formula_avg_time)

        # Columna G (Avg Memory) => 'Raw Data'!$K:$K
        formula_avg_mem = (
            f"=PROMEDIO.SI.CONJUNTO('Raw Data'!$K:$K,"
            f"'Raw Data'!$B:$B,\"{model_val}\","
            f"'Raw Data'!$C:$C,\"{temp_val}\","
            f"'Raw Data'!$D:$D,\"{top_p_val}\","
            f"'Raw Data'!$E:$E,\"{top_k_val}\")"
        )
        comp_sheet.write_formula(fila_excel, 6, formula_avg_mem)

        # Columna H (Avg Complexity) => 'Raw Data'!$O:$O
        formula_avg_comp = (
            f"=PROMEDIO.SI.CONJUNTO('Raw Data'!$O:$O,"
            f"'Raw Data'!$B:$B,\"{model_val}\","
            f"'Raw Data'!$C:$C,\"{temp_val}\","
            f"'Raw Data'!$D:$D,\"{top_p_val}\","
            f"'Raw Data'!$E:$E,\"{top_k_val}\")"
        )
        comp_sheet.write_formula(fila_excel, 7, formula_avg_comp)

        # Columna I (Config Label)
        # =A2 & " (T:" & B2 & ", p:" & C2 & ", k:" & D2 & ")"
        formula_label = (
            f"=A{fila_excel+1} & \" (T:\" & B{fila_excel+1} & \", p:\" & C{fila_excel+1} & \", k:\" & D{fila_excel+1} & \")\""
        )
        comp_sheet.write_formula(fila_excel, 8, formula_label)

        # Columna J (Rank Accuracy) => JERARQUIA.EQV(E2, $E$2:$E$<last>, 0) (mayor es mejor)
        formula_rank_acc = f"=JERARQUIA.EQV(E{fila_excel+1},$E$2:$E${total_configs+1},0)"
        comp_sheet.write_formula(fila_excel, 9, formula_rank_acc)

        # Columna K (Rank Time) => JERARQUIA.EQV(F2, $F$2:$F$<last>, 1) (menor es mejor)
        formula_rank_time = f"=JERARQUIA.EQV(F{fila_excel+1},$F$2:$F${total_configs+1},1)"
        comp_sheet.write_formula(fila_excel, 10, formula_rank_time)

        # Columna L (Rank Memory) => JERARQUIA.EQV(G2, $G$2:$G$<last>, 1)
        formula_rank_mem = f"=JERARQUIA.EQV(G{fila_excel+1},$G$2:$G${total_configs+1},1)"
        comp_sheet.write_formula(fila_excel, 11, formula_rank_mem)

        # Columna M (Rank Complexity) => JERARQUIA.EQV(H2, $H$2:$H$<last>, 1)
        formula_rank_comp = f"=JERARQUIA.EQV(H{fila_excel+1},$H$2:$H${total_configs+1},1)"
        comp_sheet.write_formula(fila_excel, 12, formula_rank_comp)

        # Columna N (Composite Rank) => PROMEDIO(J, K, L, M)
        formula_composite = f"=PROMEDIO(J{fila_excel+1},K{fila_excel+1},L{fila_excel+1},M{fila_excel+1})"
        comp_sheet.write_formula(fila_excel, 13, formula_composite)

        # Columna O (Global Best?) => SI(N2 = MIN($N$2:$N$<last>), "Yes", "")
        formula_global = f"=SI(N{fila_excel+1}=MIN($N$2:$N${total_configs+1}),\"Yes\",\"\")"
        comp_sheet.write_formula(fila_excel, 14, formula_global)

        # Columna P (Best per Model?) => SI(N2 = MIN.SI.CONJUNTO($N$2:$N$<last>, $A$2:$A$<last>, A2), "Yes", "")
        # => MIN.SI.CONJUNTO en español
        formula_best_model = (
            f"=SI(N{fila_excel+1}=MIN.SI.CONJUNTO($N$2:$N${total_configs+1},$A$2:$A${total_configs+1},A{fila_excel+1}),"
            f"\"Yes\",\"\")"
        )
        comp_sheet.write_formula(fila_excel, 15, formula_best_model)

    # Ajustar anchos
    comp_sheet.set_column(0, 0, 20)
    comp_sheet.set_column(1, 3, 14)
    comp_sheet.set_column(4, 7, 18)
    comp_sheet.set_column(8, 8, 40)
    comp_sheet.set_column(9, 15, 16)

    # =========================================================================
    # 5) HOJA "Explanation": DETALLE DE VARIABLES Y FÓRMULAS (EN ESPAÑOL)
    # =========================================================================
    expl_sheet = workbook.add_worksheet("Explanation")
    texto_explicacion = (
        "EXPLICACIÓN DE LAS FÓRMULAS EN ESPAÑOL:\n\n"
        "Hoja 'Raw Data':\n"
        " - 'Total Tests' (columna M): =H + I (true_count + false_count)\n"
        " - 'Accuracy' (columna N): =SI(M>0, H/M, \"\")\n"
        " - 'Numeric Complexity' (columna O): Se extraen los valores de 'loops: X, conditionals: Y' usando\n"
        "   funciones en español: VALOR, ESPACIOS, EXTRAE, HALLAR, LARGO.\n\n"
        "Hoja 'Comparisons':\n"
        " - 'Avg Accuracy', 'Avg Time (s)', 'Avg Memory (MB)', 'Avg Complexity':\n"
        "   se calculan con PROMEDIO.SI.CONJUNTO (filtrando por model, temperature, top_p, top_k).\n"
        " - 'Config Label': concatena model + parámetros para identificar la configuración.\n"
        " - 'Rank Accuracy': JERARQUIA.EQV(E2, rango, 0) => mayor es mejor.\n"
        " - 'Rank Time', 'Rank Memory', 'Rank Complexity': JERARQUIA.EQV(...,1) => menor es mejor.\n"
        " - 'Composite Rank': =PROMEDIO(J, K, L, M)\n"
        " - 'Global Best?': marca 'Yes' si la fila tiene el Composite Rank mínimo global => =SI(N2=MIN($N$2:$N$<last>),\"Yes\",\"\").\n"
        " - 'Best per Model?': marca 'Yes' si la fila tiene el Composite Rank mínimo dentro del mismo modelo =>\n"
        "   =SI(N2=MIN.SI.CONJUNTO($N$2:$N$<last>, $A$2:$A$<last>, A2), \"Yes\", \"\").\n\n"
        "Asegúrate de tener Excel 2019 u Office 365 en español para que existan las funciones\n"
        "MIN.SI.CONJUNTO, PROMEDIO.SI.CONJUNTO y JERARQUIA.EQV. En versiones anteriores no funcionan.\n"
    )
    expl_sheet.write(0, 0, texto_explicacion)
    expl_sheet.set_column(0, 0, 100)

    # =========================================================================
    # 6) HOJA "Charts": GRÁFICOS NATIVOS DE EXCEL (EN ESPAÑOL)
    # =========================================================================
    charts_sheet = workbook.add_worksheet("Charts")
    charts_sheet.write(0, 0, "Visualización de Métricas por Configuración (usando funciones en español)")

    last_comp_row = total_configs + 1  # La última fila con datos en Comparisons

    # --- Gráfico 1: Column Chart para Avg Accuracy ---
    chart_acc = workbook.add_chart({'type': 'column'})
    chart_acc.add_series({
        'name': "Promedio Accuracy",
        'categories': f"=Comparisons!$I$2:$I${last_comp_row}",  # Config Label
        'values':     f"=Comparisons!$E$2:$E${last_comp_row}",  # Avg Accuracy
        'data_labels': {'value': True},
    })
    chart_acc.set_title({'name': 'Promedio Accuracy por Configuración'})
    chart_acc.set_x_axis({'name': 'Configuración'})
    chart_acc.set_y_axis({'name': 'Avg Accuracy'})
    chart_acc.set_legend({'position': 'bottom'})
    charts_sheet.insert_chart('B3', chart_acc, {'x_scale': 1.2, 'y_scale': 1.2})

    # --- Gráfico 2: Column Chart para Avg Execution Time ---
    chart_time = workbook.add_chart({'type': 'column'})
    chart_time.add_series({
        'name': "Promedio Tiempo (s)",
        'categories': f"=Comparisons!$I$2:$I${last_comp_row}",
        'values':     f"=Comparisons!$F$2:$F${last_comp_row}",
        'data_labels': {'value': True},
    })
    chart_time.set_title({'name': 'Promedio Tiempo de Ejecución por Configuración'})
    chart_time.set_x_axis({'name': 'Configuración'})
    chart_time.set_y_axis({'name': 'Avg Time (s)'})
    chart_time.set_legend({'position': 'bottom'})
    charts_sheet.insert_chart('B20', chart_time, {'x_scale': 1.2, 'y_scale': 1.2})

    # --- Gráfico 3: Column Chart para Avg Memory ---
    chart_mem = workbook.add_chart({'type': 'column'})
    chart_mem.add_series({
        'name': "Promedio Memoria (MB)",
        'categories': f"=Comparisons!$I$2:$I${last_comp_row}",
        'values':     f"=Comparisons!$G$2:$G${last_comp_row}",
        'data_labels': {'value': True},
    })
    chart_mem.set_title({'name': 'Promedio de Memoria por Configuración'})
    chart_mem.set_x_axis({'name': 'Configuración'})
    chart_mem.set_y_axis({'name': 'Avg Memory (MB)'})
    chart_mem.set_legend({'position': 'bottom'})
    charts_sheet.insert_chart('B37', chart_mem, {'x_scale': 1.2, 'y_scale': 1.2})

    # --- Gráfico 4: Scatter Chart - Time vs. Accuracy (una serie por configuración) ---
    scatter_time = workbook.add_chart({'type': 'scatter'})
    for row_excel in range(2, total_configs+2):
        # Nombre de la serie: Config Label (col I)
        serie_name = f"=Comparisons!$I${row_excel}"
        # Eje X: Avg Time (col F)
        x_range = f"=Comparisons!$F${row_excel}"
        # Eje Y: Avg Accuracy (col E)
        y_range = f"=Comparisons!$E${row_excel}"
        scatter_time.add_series({
            'name':       serie_name,
            'categories': x_range,
            'values':     y_range,
            'data_labels': {'value': True},
            'marker':     {'type': 'circle', 'size': 7},
        })
    scatter_time.set_title({'name': 'Tiempo vs. Accuracy (por Configuración)'})
    scatter_time.set_x_axis({'name': 'Avg Time (s)'})
    scatter_time.set_y_axis({'name': 'Avg Accuracy'})
    scatter_time.set_legend({'position': 'bottom'})
    charts_sheet.insert_chart('K3', scatter_time, {'x_scale': 1.2, 'y_scale': 1.2})

    # --- Gráfico 5: Scatter Chart - Memory vs. Accuracy (una serie por configuración) ---
    scatter_mem = workbook.add_chart({'type': 'scatter'})
    for row_excel in range(2, total_configs+2):
        serie_name = f"=Comparisons!$I${row_excel}"
        x_range = f"=Comparisons!$G${row_excel}"  # Memory
        y_range = f"=Comparisons!$E${row_excel}"  # Accuracy
        scatter_mem.add_series({
            'name':       serie_name,
            'categories': x_range,
            'values':     y_range,
            'data_labels': {'value': True},
            'marker':     {'type': 'square', 'size': 7},
        })
    scatter_mem.set_title({'name': 'Memoria vs. Accuracy (por Configuración)'})
    scatter_mem.set_x_axis({'name': 'Avg Memory (MB)'})
    scatter_mem.set_y_axis({'name': 'Avg Accuracy'})
    scatter_mem.set_legend({'position': 'bottom'})
    charts_sheet.insert_chart('K20', scatter_mem, {'x_scale': 1.2, 'y_scale': 1.2})

    # =========================================================================
    # 7) CERRAMOS Y GUARDAMOS EL ARCHIVO EXCEL
    # =========================================================================
    workbook.close()
    print(f"Archivo Excel '{excel_filename}' creado exitosamente.")
    print("Incluye fórmulas nativas de Excel en español (SI, PROMEDIO.SI.CONJUNTO, MIN.SI.CONJUNTO, JERARQUIA.EQV, etc.).")
    print("Recuerda que estas funciones requieren Excel 2019 u Office 365 en español.")

if __name__ == "__main__":
    main()


Archivo Excel 'Analisis_XIV.xlsx' creado exitosamente.
Incluye fórmulas nativas de Excel en español (SI, PROMEDIO.SI.CONJUNTO, MIN.SI.CONJUNTO, JERARQUIA.EQV, etc.).
Recuerda que estas funciones requieren Excel 2019 u Office 365 en español.
